# Jamming transitions in invasion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/zoo/jamming.ipynb)

*Ilina O, Gritsenko PG, Syga S, Lippoldt J, La Porta CAM, Chepizhko O,
Grosser S, Vullings M, Bakker GJ, Starruß J, Bult P, Käs JA, Zapperi S,
Deutsch A, Friedl P (2020). Cell–cell adhesion and 3D matrix confinement
determine jamming transitions in breast cancer invasion. Nat Cell Biol 22:
1103–1115. [doi:10.1038/s41556-020-0552-6](https://doi.org/10.1038/s41556-020-0552-6)*

**Question.** Breast cancer cells invade as cohesive sheets, as fluid
sheets whose cells move independently, or as single cells. The experiments
of the paper vary E-cadherin (cell–cell adhesion) and the density of the
collagen matrix (confinement). What is the minimal set of interactions that
produces these modes, and how do the two parameters select among them?

The notebook reproduces the invasion modes of Fig 5d and a coarse version
of the phase diagram Fig 5e, and then, beyond the paper, lets the cells
invade from a spheroid instead of a straight sheet.

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import lgca
from lgca import zoo
from lgca.model import run_model
from lgca.study import sweep
from lgca.zoo import jamming

zoo.parameter_table("jamming")

## The model

Cells on a hexagonal lattice with volume exclusion, 6 velocity and 3 rest
channels ($K = 9$), mimic the interface assay: cells migrate between a
collagen-coated surface and a collagen network. In every step, a node's
cells take a new channel configuration $s'$ with probability
$P(s') \propto \exp E(s')$, where

$$E = E_\text{steric} + E_\text{ECM} + E_\text{aggregation} + E_\text{alignment}:$$

- **steric repulsion** $-\beta_\text{steric}\, j(s')\cdot\sum_i c_i \tilde\rho(r + c_i)$
  pushes cells away from nodes above the homeostatic density $\rho_0 = 3$
  ($\tilde\rho = (n - \rho_0)/(K - \rho_0)$ above $\rho_0$, else 0);
- **confinement** $\rho_\text{ECM}\, n_\text{rest}(s')$ makes cells rest
  where the matrix is dense;
- **aggregation** $\beta\, j(s')\cdot\nabla u$ with
  $u = n_\text{nb}(1 - n_\text{nb}/n_\text{crit})^+/(2n_\text{crit})$ keeps an
  optimal number of neighbours, $n_\text{crit} = (b+1)\rho_0$;
- **alignment** $\beta\,\big(j(s')\cdot J_\text{nb}/(2b) + n_\text{rest}(s')\,
  n_\text{rest,nb}/(b\rho_0)\big)$ makes cells move and rest with their
  neighbours.

$j(s')$ is the flux of the new configuration, $J_\text{nb}$ the flux of the
neighbouring nodes. Cells open up the matrix where they are,
$\rho_\text{ECM} \leftarrow \rho_\text{ECM}(1 - \alpha n/K)$, and new cells
flow in from the lower edge: every free channel of the two lowest rows is
filled with probability $r_b = 0.05$. The adhesion strength $\beta$ and the
initial matrix density $\bar\rho_\text{ECM}$ are the two parameters of the
phase diagram.

None of the four energy terms is built into the library; `lgca.zoo.jamming`
defines them with `@reorientation_term`, and the matrix degradation and the
influx with `@interaction`, in a few lines each. The spec lists them in the
order of the original code:

In [ ]:
spec = jamming.build_spec()
for operator in spec.dynamics.operators:
    print(operator)

## Invasion modes (Fig 5d)

A 50 × 50 lattice starts with two rows of cells at the homeostatic
density; after 50 steps of transient, 200 steps are observed, as in the
paper. The four corners of the phase diagram: weak and strong adhesion,
sparse and dense matrix. Arrows show the flux at each node, coloured by the
velocity correlation with the neighbouring nodes (red anticorrelated, green
correlated); blue circles are single cells, cells without another cell on
their node or the neighbouring nodes.

In [ ]:
WEAK, STRONG = 0.2, 10.0  # adhesion β
SPARSE, DENSE = 0.2, 5.0  # matrix ρ̄_ECM
corners = {(beta, ecm): run_model(jamming.build_spec(beta=beta, ecm=ecm), showprogress=False)
           for ecm in (SPARSE, DENSE) for beta in (STRONG, WEAK)}


def draw_mode(axis, lattice, title):
    """The flux at every node, coloured by the local velocity correlation, and the single cells."""
    state = lgca.lattice_state.LatticeState(lattice)
    x, y = lattice.xcoords, lattice.ycoords
    flux = state.flux
    moving = np.linalg.norm(flux, axis=-1) > 0
    correlation = np.nan_to_num(jamming.local_correlation(lattice))
    axis.scatter(x[state.density > 0], y[state.density > 0], s=6, color="0.85", lw=0)
    axis.quiver(x[moving], y[moving], flux[moving, 0], flux[moving, 1], correlation[moving], cmap="RdYlGn",
                clim=(-1, 1), angles="xy", scale_units="xy", scale=1.5, width=0.004)
    alone = (state.density == 1) & (state.neighbor_sum(state.density) == 0)
    axis.scatter(x[alone], y[alone], s=30, facecolors="none", edgecolors="tab:blue", lw=1.2)
    axis.set(aspect="equal", xticks=[], yticks=[], title=title)


fig, axes = plt.subplots(2, 2, figsize=(10, 10), constrained_layout=True)
for axis, ((beta, ecm), run) in zip(axes.flat, corners.items()):
    mode = jamming.invasion_mode(run)
    draw_mode(axis, run.lgca, f"β = {beta}, ρ̄_ECM = {ecm}\n{mode['single_cells']:.0f} single cells, "
                              f"correlation {mode['correlation']:.2f}")
plt.show()
plt.close(fig)

The three principal responses of the paper appear:

1. **Weak adhesion, sparse matrix** (top right): cells detach and spread as
   single cells, with little correlation: gas-like dissemination.
2. **Strong adhesion** (left column): hardly any single cells, and
   neighbouring cells move together: a collective sheet.
3. **Weak adhesion, dense matrix** (bottom right): the matrix holds the
   cells together as a sheet, but without adhesion their movements stay
   uncorrelated: a fluid sheet.

## The phase diagram (Fig 5e), coarsely

The paper varies $\beta$ and $\bar\rho_\text{ECM}$ from 0 to 10 in steps
of 0.2 with five runs each, 13 005 runs (`jamming.FULL_GRID`). A 6 × 6 grid
on a logarithmic scale with two seeds shows the same regions:

In [ ]:
values = list(np.round(np.geomspace(0.1, 10, 6), 2))
table = sweep(
    jamming.build_spec(),
    grid={"dynamics.operators[1].terms[jamming.adhesion].beta": values, "state.fields.ecm": values},
    seeds=range(2),
    measure={"single_cells": jamming.total_single_cells, "correlation": jamming.mean_correlation},
    n_jobs=4,
    showprogress=False,
)
table = table.rename(columns=dict(zip(table.columns[:2], ["beta", "ecm"])))
mean = table.groupby(["beta", "ecm"])[["single_cells", "correlation"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
for axis, name, cmap, label in ((axes[0], "single_cells", "Blues", "single cells (sum over 200 steps)"),
                                (axes[1], "correlation", "RdYlGn", "next-neighbour velocity correlation")):
    grid = mean[name].unstack("beta")  # rows: ecm, columns: beta
    image = axis.pcolormesh(np.arange(len(values) + 1), np.arange(len(values) + 1), grid.values, cmap=cmap)
    axis.set_xticks(np.arange(len(values)) + 0.5, [f"{v:g}" for v in grid.columns])
    axis.set_yticks(np.arange(len(values)) + 0.5, [f"{v:g}" for v in grid.index])
    axis.set(xlabel="adhesion strength β", ylabel="matrix density ρ̄_ECM")
    fig.colorbar(image, ax=axis, label=label)
plt.show()
plt.close(fig)

Single cells are released only where both adhesion and confinement are
weak (region 1); strong adhesion gives correlated movement (region 2), and
a dense matrix suppresses single cells even without adhesion, but not the
lack of correlation (region 3). Lowering adhesion thus fluidizes the sheet
in two steps: first to an uncorrelated fluid sheet, and, if the matrix is
sparse, to single cells.

## Beyond the paper: invasion from a spheroid

In the interface assay of the paper the cells come from a spheroid above
the imaged plane, which the model idealizes as a straight sheet. Here the
spheroid is in the plane: a disc of cells at the homeostatic density in
the centre of an 80 × 80 lattice, which keeps supplying cells, invades a
uniform matrix in all directions (`build_spec(setup="spheroid")`):

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10), constrained_layout=True)
for axis, (beta, ecm) in zip(axes.flat, [(STRONG, SPARSE), (WEAK, SPARSE), (STRONG, DENSE), (WEAK, DENSE)]):
    run = run_model(jamming.build_spec(beta=beta, ecm=ecm, setup="spheroid"), showprogress=False)
    mode = jamming.invasion_mode(run)
    draw_mode(axis, run.lgca, f"spheroid, β = {beta}, ρ̄_ECM = {ecm}\n{mode['single_cells']:.0f} single cells, "
                              f"correlation {mode['correlation']:.2f}")
plt.show()
plt.close(fig)

The same three modes, now as a disseminating halo of single cells, a
cohesive disc with collectively moving cells, and a compact, uncorrelated
disc in a dense matrix. The matrix field (`ecm`) records where the cells
have opened up space; the explorer below can show it.

## Differences from the paper

- **The adhesion potential** follows the model definition of the
  supplement, $u = n_\text{nb}(1 - n_\text{nb}/n_\text{crit})^+/(2n_\text{crit})$;
  the original code used four times that value (`build_spec(adhesion_scale=4)`).
  It matters: with the definition's potential, single cells are released
  up to $\beta \approx 10$ in a sparse matrix (520 over 200 steps at
  $\beta = 10$, two seeds), with the code's they stop by $\beta \approx 3$
  (161 at $\beta = 3$, 16 at $\beta = 5$). The paper's region 2 covers
  most of its range $0 \le \beta \le 10$, as with the code's potential;
  with the definition's, the same regions appear at larger $\beta$.
- **Phase diagram.** 6 × 6 values on a logarithmic grid with two seeds
  instead of 51 × 51 with five; `jamming.FULL_GRID` holds the paper's grid
  (about three hours on one core).
- **The spheroid** is our addition.

## Explore it

The sheet, with the matrix density as a view: move $\beta$ while the
cells invade, or restart with another matrix density:

In [ ]:
explorer = lgca.explore(
    jamming.build_spec(beta=1.0, ecm=1.0),
    {"dynamics.operators[1].terms[jamming.adhesion].beta": (0.0, 10.0), "state.fields.ecm": (0.0, 10.0)},
    view="flux",
    measure={"single cells": jamming.single_cells, "correlation": jamming.velocity_correlation},
)
explorer

## Things to try

1. **Speed at the front (Fig 5c).** Measure how far the front advances in
   200 steps as a function of $\bar\rho_\text{ECM}$ for weak and strong
   adhesion.
2. **Aggregation or alignment alone.** `jamming.adhesion` contains both
   adhesive effects; split it into two terms (copy the function) and give
   them different strengths. Which one prevents single cells?
3. **Matrix degradation.** Set $\alpha = 0$: the cells cannot open the
   gap. How does the phase diagram change?
4. **A heterogeneous matrix.** Give `state.fields.ecm` a random array
   instead of a number, e.g. collagen fibres as stripes of high density.